# 🏆 Zalo AI Challenge 2026 — DataMate High-Score Inference Engine (Production Ready)

**Mô hình:** `Qwen2.5-Coder-7B-Instruct` + Fine-Tuned LoRA Adapter
**Mục tiêu:** Sinh tệp dự đoán chuẩn `submission.jsonl` cho toàn bộ 1.000 câu hỏi Public Test
**Trang bị đặc biệt:**
* ✅ **Auto Header Detection:** Tự động phát hiện và căn chỉnh tiêu đề bảng Excel lệch ở Dòng 1, Dòng 2, Dòng 3.
* ✅ **VN Currency & Number Cleaner:** Tự động chuẩn hóa số liệu tiếng Việt (`79k` -> `79000`, `69.000đ` -> `69000`).
* ✅ **Zero-Hang Sandbox Engine:** Bộ ngắt Timeout 8 giây (`signal.alarm(8)`) chống treo vĩnh viễn.
* ✅ **Penalized Accuracy Defense:** Tự động bảo toàn điểm số bằng cơ chế `abstain` (0 điểm) thay vì đoán mò (-1 điểm).
* ✅ **Live Progress Logging:** Hiển thị tức thời tiến trình từng câu (`flush=True`), không bị nghẽn đệm.


In [ ]:
# 1. Cài đặt Unsloth và các thư viện cần thiết
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install xformers triton pandas openpyxl


In [ ]:
# 2. Tự động kiểm tra & nạp Checkpoint LoRA Adapter
import os, zipfile, glob
from pathlib import Path

# Thử kết nối Google Drive (nếu người dùng muốn dùng Drive, không bắt buộc)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception:
    pass

LORA_DIR = None

# 1. Kiểm tra nếu thư mục adapter đã có sẵn trên Colab
if os.path.exists('/content/lora_adapter/adapter_model.safetensors'):
    LORA_DIR = '/content/lora_adapter'
elif os.path.exists('/content/datamate_lora_adapter/adapter_model.safetensors'):
    LORA_DIR = '/content/datamate_lora_adapter'
else:
    # 2. Quét tìm file zip lora adapter trên Colab hoặc Drive
    found_zips = glob.glob('/content/**/datamate_lora_adapter*.zip', recursive=True) or \
                 glob.glob('/content/**/*adapter*.zip', recursive=True) or \
                 glob.glob('/content/drive/**/datamate_lora_adapter*.zip', recursive=True)
    
    if found_zips:
        zip_file = found_zips[0]
        print(f'📦 Đang giải nén LoRA adapter từ: {zip_file}...')
        with zipfile.ZipFile(zip_file, 'r') as zf:
            zf.extractall('/content')
        if os.path.exists('/content/lora_adapter'):
            LORA_DIR = '/content/lora_adapter'
        else:
            found_safetensors = glob.glob('/content/**/adapter_model.safetensors', recursive=True)
            if found_safetensors:
                LORA_DIR = str(Path(found_safetensors[0]).parent)
    else:
        # 3. Kiểm tra các đường dẫn Drive dự kiến
        for d in [
            '/content/drive/MyDrive/zalo_2026_datamate_model/lora_adapter',
            '/content/drive/MyDrive/zalo_datamate_lora',
            '/content/drive/MyDrive/lora_adapter',
            'checkpoints/datamate/lora_adapter'
        ]:
            if os.path.exists(d):
                LORA_DIR = d
                break

if not LORA_DIR or not os.path.exists(LORA_DIR):
    print('📂 Không tìm thấy adapter sẵn. Vui lòng bấm nút "Choose Files" bên dưới để chọn file datamate_lora_adapter.zip:')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/lora_adapter'):
                LORA_DIR = '/content/lora_adapter'
                break

print(f'✅ Đã tìm thấy và sẵn sàng LoRA Adapter tại: {LORA_DIR}')



In [ ]:
# 3. Tự động kiểm tra, nạp & giải nén datamate_public_test.zip (61 MB)
import os, zipfile, glob
from pathlib import Path

# Kiểm tra nếu đã giải nén rồi
if os.path.exists('/content/public_test/questions.jsonl'):
    print('✅ Dữ liệu public test đã có sẵn tại /content/public_test!')
else:
    # Quét tìm file zip trên Colab hoặc Drive
    found_zips = glob.glob('/content/**/datamate_public_test*.zip', recursive=True) or \
                 glob.glob('/content/drive/**/datamate_public_test*.zip', recursive=True)
    
    if found_zips:
        zip_file = found_zips[0]
        print(f'📦 Đang giải nén từ tệp có sẵn: {zip_file}...')
        with zipfile.ZipFile(zip_file, 'r') as zf:
            zf.extractall('/content/public_test')
        print('✅ Giải nén thành công!')
    else:
        print('📂 Vui lòng bấm nút "Choose Files" bên dưới để chọn datamate_public_test.zip:')
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall('/content/public_test')
                print(f'✅ Đã giải nén {fname} vào /content/public_test!')
                break

print(f"📊 Dữ liệu: {len(os.listdir('/content/public_test/data'))} thư mục doanh nghiệp sẵn sàng!")



In [ ]:
# 4. Nạp Model Qwen2.5-Coder-7B ở chế độ 4-bit và gắn LoRA Adapter
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

print(f'Đang nạp mô hình và LoRA Adapter từ: {LORA_DIR}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

# Kích hoạt chế độ suy luận siêu tốc (tăng tốc gấp 2 lần, tiết kiệm VRAM)
FastLanguageModel.for_inference(model)
print('✅ Mô hình đã sẵn sàng cho suy luận!')


In [ ]:
# 5. Full 5-Pillar Agentic Utilities, Smart Header Loader & Sandbox with Timeout
import json, re, math, sqlite3, datetime, time, io, contextlib, ast, signal
from pathlib import Path
import pandas as pd
import numpy as np

def resolve_time(query, as_of_date):
    try:
        curr_dt = datetime.datetime.strptime(as_of_date.strip(), '%Y-%m-%d').date()
    except Exception:
        curr_dt = datetime.date.today()
    q = query.lower()
    if 'hôm nay' in q: return f'hôm nay ({curr_dt.isoformat()})'
    if 'hôm qua' in q: return f'hôm qua ({(curr_dt - datetime.timedelta(days=1)).isoformat()})'
    if 'tháng này' in q: return f'tháng này ({curr_dt.month}/{curr_dt.year})'
    if 'tháng trước' in q:
        m = 12 if curr_dt.month == 1 else curr_dt.month - 1
        y = curr_dt.year - 1 if curr_dt.month == 1 else curr_dt.year
        return f'tháng trước ({m}/{y})'
    return f'ngày neo ({as_of_date})'

def infer_type(val):
    if val is None: return 'abstain', None
    if isinstance(val, (pd.Series, pd.DataFrame)):
        if len(val) == 0: return 'abstain', None
        val = val.iloc[0] if isinstance(val, pd.Series) else val.iloc[0, 0]
    if hasattr(val, 'item') and callable(val.item):
        try: val = val.item()
        except Exception: pass
    if isinstance(val, (bool, np.bool_)): return 'boolean', bool(val)
    if isinstance(val, (int, float, np.integer, np.floating)):
        if math.isnan(float(val)): return 'abstain', None
        if math.isclose(float(val), round(float(val)), abs_tol=1e-5): return 'number', int(round(float(val)))
        return 'number', round(float(val), 2)
    if isinstance(val, list): return 'list', val
    if isinstance(val, dict) and 'columns' in val: return 'table', val
    try: val_str = str(val).strip()
    except Exception: return 'abstain', None
    if val_str in ('__ABSTAIN__', '#VALUE!', 'NaN', 'nan', 'None', ''): return 'abstain', None
    if val_str == '__UNANSWERABLE__': return 'unanswerable', None
    if val_str.endswith('%'):
        try: return 'percent', round(float(val_str[:-1].replace(',', '.')), 2)
        except Exception: pass
    if re.match(r'^\d{4}-\d{2}-\d{2}$', val_str): return 'date', val_str
    try:
        clean_num = val_str.replace(' ', '').replace(',', '.')
        f_num = float(clean_num)
        if math.isclose(f_num, round(f_num), abs_tol=1e-5): return 'number', int(round(f_num))
        return 'number', round(f_num, 2)
    except Exception: pass
    return 'string', val_str

def auto_detect_header(df_raw):
    """Tự động phát hiện dòng tiêu đề thực sự (giải quyết triệt để lỗi header ở dòng 1, 2, 3)"""
    keywords = ['ngày', 'mã', 'tên', 'stt', 'tiền', 'sl', 'số lượng', 'giá', 'hàng', 'khách', 'đvt', 'trạng thái', 'loại', 'imei', 'sđt', 'nợ', 'đg', 'tt', 'chi', 'nhập', 'doanh thu', 'bán', 'đơn']
    best_row = 0
    best_score = -1
    for r in range(min(8, len(df_raw))):
        row_vals = [str(v).lower().strip() for v in df_raw.iloc[r] if pd.notna(v)]
        if not row_vals: continue
        kw_matches = sum(1 for v in row_vals if any(k in v for k in keywords))
        str_count = sum(1 for v in row_vals if not v.replace('.', '').replace('-', '').isdigit())
        score = kw_matches * 4 + str_count
        if score > best_score and (kw_matches >= 2 or (kw_matches >= 1 and str_count >= 3)):
            best_score = score
            best_row = r
    return best_row

def is_date_column(col_name, s=None):
    """Xác định cột có phải cột ngày tháng / thời gian hay không để bảo vệ tuyệt đối."""
    name_lower = str(col_name).lower()
    keywords = ['ngày', 'date', 'thời gian', 'time', 'tháng', 'kỳ', 'period', 'nam', 'năm']
    if any(k in name_lower for k in keywords):
        return True
    if s is not None and isinstance(s, pd.Series):
        sample = s.dropna().astype(str).head(10)
        date_pattern_cnt = 0
        for v in sample:
            v_str = str(v).strip()
            if '/' in v_str or ('-' in v_str and len(v_str) >= 8):
                date_pattern_cnt += 1
        if len(sample) > 0 and (date_pattern_cnt / len(sample) >= 0.5):
            return True
    return False

def clean_numeric_series(s, col_name=''):
    """Chuẩn hóa số liệu Việt Nam kèm đơn vị: 79k -> 79000, 1 ly -> 1.0, 10 cây -> 10.0, 69.000đ -> 69000"""
    if not isinstance(s, pd.Series) or len(s) == 0: return s
    if is_date_column(col_name, s): return s
    is_qty_or_money = any(k in col_name.lower() for k in ['số lượng', 'sl', 'giá', 'tiền', 'doanh thu', 'chi', 'nhập', 'bán', 'tồn', 'nợ', 'thành tiền', 'đơn giá', 'tổng'])
    def parse_num(val):
        if pd.isna(val) or val is None: return np.nan
        if isinstance(val, (int, float, np.number)): return float(val)
        v = str(val).strip().lower()
        if not v or v in ('nan', 'none', '#value!', '#n/a', 'nan%'): return np.nan
        m = re.match(r'^([+-]?[\d\.,\s]+)', v)
        if m:
            num_str = m.group(1).replace(' ', '')
            if ',' in num_str and '.' in num_str: num_str = num_str.replace(',', '')
            elif '.' in num_str: num_str = re.sub(r'\.(\d{3})(?!\d)', r'\1', num_str)
            elif ',' in num_str: num_str = re.sub(r',(\d{3})(?!\d)', r'\1', num_str).replace(',', '.')
            try:
                val_float = float(num_str)
                if 'k' in v and not v.startswith('k'): val_float *= 1000.0
                elif 'tr' in v or 'm' in v: val_float *= 1000000.0
                return val_float
            except Exception: pass
        return np.nan if is_qty_or_money else val
    sample = s.dropna().astype(str).head(20)
    if len(sample) == 0: return s
    converted = [parse_num(x) for x in sample]
    num_cnt = sum(1 for x in converted if isinstance(x, (int, float, np.number)) and not np.isnan(x))
    if is_qty_or_money or (num_cnt / len(sample) >= 0.5):
        cleaned = s.map(parse_num)
        return pd.to_numeric(cleaned, errors='coerce')
    return s

def clean_date_series(s, col_name=''):
    """Giải mã thông minh mọi định dạng ngày: DD/MM/YYYY, YYYY-MM-DD, Excel serial (46143) kèm ffill"""
    if not isinstance(s, pd.Series) or len(s) == 0: return s
    def smart_date_parser(val):
        if pd.isna(val) or val is None: return None
        if isinstance(val, (pd.Timestamp, pd.DatetimeIndex)): return val.strftime('%Y-%m-%d')
        v_str = str(val).strip()
        if not v_str or v_str.lower() in ('nan', 'none', 'nat', '#value!', '#n/a'): return None
        try:
            f = float(v_str)
            if 30000 < f < 60000:
                dt = pd.to_datetime(f, unit='D', origin='1899-12-30')
                return dt.strftime('%Y-%m-%d')
        except Exception: pass
        if len(v_str) >= 10 and v_str[:4].isdigit() and v_str[4] in ('-', '/'):
            try:
                dt = pd.to_datetime(v_str, errors='coerce')
                if pd.notna(dt): return dt.strftime('%Y-%m-%d')
            except Exception: pass
        try:
            dt = pd.to_datetime(v_str, dayfirst=True, errors='coerce')
            if pd.notna(dt): return dt.strftime('%Y-%m-%d')
        except Exception: pass
        return v_str
    return s.map(smart_date_parser).ffill()

def load_biz_tables(biz_folder):
    """Đọc dữ liệu thông minh: tách biệt cột ngày và số, chống lỗi header và vỡ ngày tháng"""
    p = Path(biz_folder)
    tables = {}
    for f in sorted(p.glob('*')):
        if f.name == 'README.md': continue
        stem = re.sub(r'\W|^(?=\d)', '_', f.stem)
        ext = f.suffix.lower()
        if ext == '.csv':
            try:
                try: df_raw = pd.read_csv(f, header=None, nrows=8, encoding='utf-8-sig', low_memory=False)
                except Exception: df_raw = pd.read_csv(f, header=None, nrows=8, encoding='latin1', low_memory=False)
                h = auto_detect_header(df_raw)
                try: df = pd.read_csv(f, header=h, encoding='utf-8-sig', low_memory=False)
                except Exception: df = pd.read_csv(f, header=h, encoding='latin1', low_memory=False)
                df.columns = [str(c).strip() for c in df.columns]
                for c in df.columns:
                    col_str = str(c)
                    if is_date_column(col_str, df[c]):
                        df[c] = clean_date_series(df[c], col_name=col_str)
                    else:
                        if not df[c].dtype.kind in ('i', 'f'):
                            df[c] = clean_numeric_series(df[c], col_name=col_str)
                tables[stem] = df
            except Exception: pass
        elif ext in ('.xlsx', '.xls'):
            try:
                xl = pd.ExcelFile(f)
                for s in xl.sheet_names:
                    df_raw = pd.read_excel(xl, sheet_name=s, header=None)
                    h = auto_detect_header(df_raw)
                    df = df_raw.iloc[h+1:].copy().reset_index(drop=True)
                    df.columns = [str(c).strip() for c in df_raw.iloc[h].values]
                    for c in df.columns:
                        col_str = str(c)
                        if is_date_column(col_str, df[c]):
                            df[c] = clean_date_series(df[c], col_name=col_str)
                        else:
                            if not df[c].dtype.kind in ('i', 'f'):
                                df[c] = clean_numeric_series(df[c], col_name=col_str)
                    sheet_clean = re.sub(r'\W|^(?=\d)', '_', s)
                    tables[f'{stem}_{sheet_clean}'] = df
            except Exception: pass
        elif ext in ('.sqlite', '.db'):
            try:
                conn = sqlite3.connect(f)
                cur = conn.cursor()
                cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
                for (tname,) in cur.fetchall():
                    tc = re.sub(r'\W|^(?=\d)', '_', tname)
                    df = pd.read_sql_query(f'SELECT * FROM {tname}', conn)
                    for c in df.columns:
                        col_str = str(c)
                        if is_date_column(col_str, df[c]):
                            df[c] = clean_date_series(df[c], col_name=col_str)
                        else:
                            if not df[c].dtype.kind in ('i', 'f'):
                                df[c] = clean_numeric_series(df[c], col_name=col_str)
                    tables[f'{stem}_{tc}'] = df
                conn.close()
            except Exception: pass
    return tables

def build_compact_schema(tables):
    lines = []
    for tname, df in tables.items():
        cols = list(df.columns)[:12]
        sample = df.head(2).to_dict(orient='records')
        lines.append(f"- Bảng `{tname}` ({len(df)} dòng):\n  Cột: {cols}\n  Mẫu: {json.dumps(sample, ensure_ascii=False, default=str)}")
    return '\n'.join(lines)

class SafePandas:
    def __init__(self, real_pd, tbls):
        self._pd = real_pd
        self._tbls = tbls
    def __getattr__(self, item): return getattr(self._pd, item)
    def read_csv(self, filepath, *args, **kwargs):
        stem = Path(str(filepath)).stem
        if stem in self._tbls: return self._tbls[stem]
        for k in self._tbls:
            if stem.lower() in k.lower() or k.lower() in stem.lower(): return self._tbls[k]
        return self._pd.read_csv(filepath, *args, **kwargs)
    def read_excel(self, filepath, *args, **kwargs):
        stem = Path(str(filepath)).stem
        if stem in self._tbls: return self._tbls[stem]
        for k in self._tbls:
            if stem.lower() in k.lower() or k.lower() in stem.lower(): return self._tbls[k]
        return self._pd.read_excel(filepath, *args, **kwargs)
    def to_datetime(self, arg, *args, **kwargs):
        try: return self._pd.to_datetime(arg, *args, **kwargs)
        except Exception:
            kwargs.pop('format', None)
            kwargs['dayfirst'] = True
            kwargs['errors'] = 'coerce'
            return self._pd.to_datetime(arg, *args, **kwargs)

class TimeoutError(Exception): pass
def _sig_handler(signum, frame):
    raise TimeoutError("Code execution took longer than 8s limit")

def execute_sandbox(code_str, tables, as_of_date):
    if not code_str: return False, None
    m = re.findall(r'```python\s*(.*?)\s*```', code_str, re.DOTALL)
    clean_code = m[-1].strip() if m else re.sub(r'```', '', code_str).strip()
    lines = [
        l for l in clean_code.split('\n')
        if not l.strip().startswith(('Dưới đây', 'Để trả lời', 'Here is', 'Note:', 'Giải thích', 'import sys', 'sys.exit', 'import pandas', 'import numpy'))
    ]
    clean_code = '\n'.join(lines)
    
    globs = {
        'pd': SafePandas(pd, tables),
        'np': np,
        'math': math,
        're': re,
        'datetime': datetime,
        'tables': tables,
        'as_of_date': as_of_date,
        'result': None,
        'res': None,
        'answer': None,
        'ans': None,
    }
    if tables:
        first_key = next(iter(tables.keys()))
        globs['df'] = tables[first_key]
        for tname, tdf in tables.items():
            globs[tname] = tdf
            clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', tname)
            globs[clean_name] = tdf
            
    stdout_buf = io.StringIO()
    has_alarm = hasattr(signal, 'SIGALRM')
    if has_alarm:
        signal.signal(signal.SIGALRM, _sig_handler)
        signal.alarm(8)
    try:
        parsed = ast.parse(clean_code)
        exec(compile(parsed, '<sandbox>', 'exec'), globs)
        
        val = globs.get('result') or globs.get('res') or globs.get('answer') or globs.get('ans')
        if val is None:
            for k, v in reversed(list(globs.items())):
                if k not in ('pd', 'np', 'math', 're', 'datetime', 'tables', 'as_of_date', 'df') and not k.startswith('_'):
                    if isinstance(v, (int, float, str, bool, list, dict, pd.Series, pd.DataFrame)) and v is not None:
                        val = v
                        break
        return True, val
    except Exception as e:
        return False, str(e)
    finally:
        if has_alarm:
            signal.alarm(0)



In [ ]:
# 6. Ultra-Fast High-Accuracy Inference with Live Logging & Auto-Resume Engine
import os, zipfile, glob, time, json
from pathlib import Path
from transformers import StoppingCriteria, StoppingCriteriaList

# --- 1. TỰ ĐỘNG TÌM FILE DỮ LIỆU ---
found_q = glob.glob('/content/**/questions.jsonl', recursive=True)
if found_q:
    q_file = found_q[0]
    biz_root = Path(q_file).parent / 'data'
    print(f"✅ Đã tìm thấy file câu hỏi tại: {q_file}", flush=True)
    print(f"✅ Thư mục data: {biz_root}", flush=True)
else:
    found_zips = glob.glob('/content/**/datamate_public_test*.zip', recursive=True) or glob.glob('/content/**/*.zip', recursive=True)
    if found_zips:
        with zipfile.ZipFile(found_zips[0], 'r') as zf: zf.extractall('/content/public_test')
        q_file = '/content/public_test/questions.jsonl'
        biz_root = Path('/content/public_test/data')
    else:
        raise FileNotFoundError("Không tìm thấy tệp questions.jsonl trên Colab.")

# --- 2. KIỂM TRA TRẠNG THÁI MODEL ---
if 'model' not in globals() or 'tokenizer' not in globals():
    raise RuntimeError("❌ Model chưa được nạp vào RAM. Hãy bấm chạy lại Cell Load Model trước nhé!")

output_file = '/content/submission.jsonl'
with open(q_file, encoding='utf-8') as f:
    questions = [json.loads(line) for line in f if line.strip()]

# KÍCH HOẠT CHẾ ĐỘ RESUME (CHỐNG MẤT DỮ LIỆU KHI COLAB RESET)
existing_map = {}
if os.path.exists(output_file):
    with open(output_file, encoding='utf-8') as f_in:
        for line in f_in:
            if line.strip():
                try:
                    rec = json.loads(line)
                    existing_map[rec['id']] = rec
                except Exception: pass
    if existing_map:
        print(f"🔄 Tìm thấy {len(existing_map)}/{len(questions)} câu đã hoàn thành từ trước. Kích hoạt Resume!", flush=True)

submissions = list(existing_map.values())
print(f'\n🚀 Starting high-precision inference on {len(questions)} questions...\n', flush=True)

biz_cache = {}
start_time = time.time()

def get_top_tables(query, tables, top_k=4):
    if not tables or len(tables) <= top_k: return tables
    q_lower = query.lower()
    multi_indicators = ["3 tháng", "nhiều tháng", "các tháng", "từ đầu năm", "cùng kỳ", "quý", "năm 2025", "năm 2026"]
    effective_k = min(len(tables), top_k + 2 if any(m in q_lower for m in multi_indicators) else top_k)
    q_words = set(re.findall(r'\w+', q_lower))
    scores = []
    for tname, df in tables.items():
        score = sum(3 for w in q_words if w in tname.lower())
        for c in df.columns: score += sum(1 for w in q_words if w in str(c).lower())
        scores.append((score, tname))
    scores.sort(reverse=True)
    selected = [tname for _, tname in scores[:effective_k]]
    return {t: tables[t] for t in selected} if selected else tables

class FastCodeStoppingCriteria(StoppingCriteria):
    def __init__(self, tok, s_len):
        self.tok = tok
        self.s_len = s_len
    def __call__(self, input_ids, scores, **kwargs):
        gen = input_ids[0, self.s_len:]
        if len(gen) > 20:
            tail = self.tok.decode(gen[-6:], skip_special_tokens=False)
            if "```" in tail or "<|im_end|>" in tail:
                return True
        return False

with open(output_file, 'a', encoding='utf-8') as out_f:
    for i, q in enumerate(questions):
        qid = q['id']
        if qid in existing_map:
            continue

        biz_id = q.get('business_id', '')
        query = q.get('question', '')
        as_of_date = q.get('as_of_date', '2026-09-07')

        if biz_id not in biz_cache:
            biz_path = biz_root / biz_id
            if len(biz_cache) < 5:
                print(f"📁 [Nạp dữ liệu] Doanh nghiệp {biz_id}...", flush=True)
            biz_cache[biz_id] = load_biz_tables(biz_path) if biz_path.exists() else {}
        full_tables = biz_cache[biz_id]

        tables = get_top_tables(query, full_tables, top_k=4)
        schema_summary = build_compact_schema(tables)
        thint = resolve_time(query, as_of_date)
        user_prompt = (
            f'Doanh nghiệp: {biz_id}\n'
            f'Ngày neo (as_of_date): {as_of_date} ({thint})\n'
            f'Schema các bảng dữ liệu sẵn có (trong từ điển `tables` hoặc biến `df`):\n{schema_summary}\n\n'
            f'Câu hỏi: "{query}"\n\n'
            'Viết mã Python pandas lưu kết quả cuối cùng vào biến `result` (đặt trong ```python ... ```):'
        )

        messages = [
            {'role': 'system', 'content': 'Bạn là trợ lý phân tích dữ liệu DataMate. Chỉ trả về một khối mã Python duy nhất trong ```python ... ``` để tính toán kết quả và gán vào biến `result`. Tuyệt đối không thêm văn bản giải thích.'},
            {'role': 'user', 'content': user_prompt}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to('cuda')
        start_len = inputs['input_ids'].shape[1]
        stopping_criteria = StoppingCriteriaList([FastCodeStoppingCriteria(tokenizer, start_len)])

        import torch
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                use_cache=True,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=stopping_criteria,
            )
        resp = tokenizer.decode(outputs[0][start_len:], skip_special_tokens=True)

        ok, val = execute_sandbox(resp, full_tables, as_of_date)
        if not ok or val is None:
            ans_type, ans_val = 'abstain', None
        else:
            ans_type, ans_val = infer_type(val)
            if ans_val in ('#VALUE!', 'NaN', 'nan', None, ''):
                ans_type, ans_val = 'abstain', None

        record = {'id': qid, 'type': ans_type}
        if ans_type not in ('abstain', 'unanswerable'):
            record['answer'] = ans_val
        submissions.append(record)

        out_f.write(json.dumps(record, ensure_ascii=False) + '\n')
        out_f.flush()

        if len(submissions) <= 3:
            print(f'✅ [Xong câu {len(submissions)}/{len(questions)}] Q: "{query}"', flush=True)
            print(f'   Mã sinh ra:\n{resp}', flush=True)
            print(f'   Kết quả: {record}\n', flush=True)

        done_cnt = len(submissions)
        if done_cnt % 10 == 0 or done_cnt == len(questions):
            elapsed = time.time() - start_time
            processed_now = done_cnt - len(existing_map)
            if processed_now > 0:
                speed = elapsed / processed_now
                rem = (len(questions) - done_cnt) * speed / 60
                valid_cnt = sum(1 for s in submissions if s['type'] != 'abstain')
                print(f'🔥 Tiến độ: {done_cnt}/{len(questions)} câu | Tốc độ: {speed:.2f}s/câu | Còn: ~{rem:.1f} phút | Đã giải: {valid_cnt}/{done_cnt}', flush=True)

print(f'\n🎉 HOÀN THÀNH TOÀN BỘ 1.000 CÂU! File lưu tại: {output_file}', flush=True)



In [ ]:
# 7. Kiểm tra tính hợp lệ theo chuẩn Zalo AI Challenge & Tải file kết quả về máy
import os, json, shutil
from pathlib import Path
from google.colab import files

output_file = Path('/content/submission.jsonl')
DATAMATE_TYPES = {"number", "percent", "string", "date", "boolean", "list", "table", "unanswerable", "abstain"}

print("=" * 65)
print("🔍 ĐANG KIỂM TRA TÍNH HỢP LỆ THEO CHUẨN ĐỀ BÀI ZALO AI CHALLENGE 2026...")
print("=" * 65)

errors = []
line_count = 0
seen_ids = set()

if output_file.exists():
    with open(output_file, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            line_count += 1
            try:
                obj = json.loads(line)
            except Exception as e:
                errors.append(f"Dòng {line_no}: JSON không hợp lệ ({e})")
                continue
            qid = obj.get("id")
            if not qid: errors.append(f"Dòng {line_no}: Thiếu trường 'id'")
            elif qid in seen_ids: errors.append(f"Dòng {line_no}: Trùng lặp id '{qid}'")
            else: seen_ids.add(qid)
            
            qtype = obj.get("type")
            if not qtype or qtype not in DATAMATE_TYPES:
                errors.append(f"Dòng {line_no}: Kiểu dữ liệu không hợp lệ '{qtype}'")
            if qtype not in ("abstain", "unanswerable") and "answer" not in obj:
                errors.append(f"Dòng {line_no}: Kiểu '{qtype}' bắt buộc phải có trường 'answer'")
            if len(errors) > 20: break

if not errors and line_count == 1000:
    print(f"🏆 CHÚC MỪNG: FILE HỢP LỆ HOÀN TOÀN 100%! ĐỦ 1.000/1.000 CÂU!")
elif not errors and line_count > 0:
    print(f"✅ HỢP LỆ: Đã xác thực {line_count} dòng hợp lệ 100% theo schema Zalo.")
else:
    print(f"⚠️ Phát hiện {len(errors)} điểm cần lưu ý:")
    for err in errors[:10]: print(f"  • {err}")

print("=" * 65)

# Sao lưu lên Google Drive nếu có
try:
    drive_out = '/content/drive/MyDrive/zalo_2026_datamate_model/submission.jsonl'
    Path(drive_out).parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(str(output_file), drive_out)
    print(f'💾 Đã lưu bản sao dự phòng tại Google Drive: {drive_out}')
except Exception:
    pass

# Tự động tải file về máy tính qua trình duyệt
print('📥 Đang tải tệp submission.jsonl về máy tính của bạn...')
files.download(str(output_file))
print('🎉 Hoàn tất! Bạn hãy gửi tệp submission.jsonl này cho chủ dự án để nộp bài thi.')

